## Dataset Description — Diabetes Dataset

**Name:** Salik Ram Bhandari
**Student ID:** 3260364
**Module:** CN7030 — Lecture 11, Databricks Practice Lab

### Overview
The dataset used in this lab is the **Diabetes Dataset**, a well-known benchmark dataset in machine learning for binary classification tasks. It contains diagnostic health measurements for female patients aged 21 and above, and is used to predict whether a patient shows signs of diabetes.

### Structure
- **Rows:** 768 patient records
- **Columns:** 9 (8 input features + 1 target label)

### Features

| Column | Description |
|---|---|
| `Pregnancies` | Number of times pregnant |
| `Glucose` | Plasma glucose concentration (2-hour oral glucose tolerance test) |
| `BloodPressure` | Diastolic blood pressure (mm Hg) |
| `SkinThickness` | Triceps skinfold thickness (mm) |
| `Insulin` | 2-hour serum insulin (mu U/ml) |
| `BMI` | Body mass index (weight in kg / height in m²) |
| `DiabetesPedigreeFunction` | A function scoring likelihood of diabetes based on family history |
| `Age` | Age in years |
| `Outcome` | Target label: 0 = non-diabetic, 1 = diabetic |

### Class Distribution
- **Outcome = 0 (non-diabetic):** 500 patients
- **Outcome = 1 (diabetic):** 268 patients

### Data Quality Notes
Several columns (`Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`) contain biologically implausible **zero values**, which almost certainly represent missing or unrecorded measurements rather than true zeros (e.g., a blood pressure of 0 is not physiologically possible). Notably, `Insulin` has 374 zero entries — nearly half the dataset. In this lab, these zeros are replaced with the **median** of each respective column to reduce the impact of missing data on model training.

### Purpose in This Lab
The cleaned dataset is used to train a **Gaussian Naive Bayes classifier** (via PySpark MLlib) to predict diabetes onset (`Outcome`) from the 8 clinical features, with model performance tracked using **MLflow** (AUC, accuracy, F1-score) and evaluated via a confusion matrix.

In [0]:

display(spark.range(10))  # should show a table of numbers 0–9

id
0
1
2
3
4
5
6
7
8
9


In [0]:
from pyspark.sql.functions import col, when, count
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import NaiveBayes
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator)
import mlflow, mlflow.spark

df = spark.read.csv(
    "/Workspace/Users/eadricbhandari432@gmail.com/week 11/diabetes.csv",   # adjust to match your uploaded filename/path
    header=True, inferSchema=True)

print(f"Rows:{df.count()} Cols:{len(df.columns)}")
display(df.limit(5))

Rows:3045570 Cols:12


id,age,sex,bmi,bp,tc,ldl,hdl,tch,ltg,glu,progression
0,22,Male,20.33,106.25,140.89,-0.45,62.83,0.77,-0.54,130.98,29.83
1,41,Female,25.99,132.23,129.64,-1.11,37.26,0.81,1.64,151.19,46.66
2,51,Female,32.76,127.0,220.36,-1.69,49.56,0.41,-0.88,176.22,59.97
3,26,Male,35.87,138.4,194.19,-0.04,55.57,0.45,-1.38,125.32,42.44
4,42,Female,21.5,122.33,275.79,1.19,63.64,0.54,-0.69,184.72,49.36


In [0]:
# How many diabetic vs non-diabetic?
display(df_clean.groupBy("Outcome").count())

# Check columns where 0 indicates a missing/invalid value
null_cols = ["glu", "bp", "bmi"]
display(df.select([
    count(when(col(c) == 0, c)).alias(c)
    for c in null_cols]))

display(df.describe())  # click chart icon for histograms

Outcome,count
1.0,1530996
0.0,1514574


glu,bp,bmi
45,0,0


summary,id,age,sex,bmi,bp,tc,ldl,hdl,tch,ltg,glu,progression
count,3045570,3045570,3045570,3045570,3045570,3045570,3045570,3045570,3045570,3045570,3045570,3045570
mean,1522784.5,39.50258309610352,null,30.00592173550445,120.00615839727791,200.08807121819584,-3.411315451623054E-4,49.99375653818439,0.500129588221578,2.1314565089622747E-4,99.98413215916908,49.72972087655145
stddev,879180.4740054798,10.003199287326046,null,6.99445817541528,9.998368392676527,49.99808330880224,1.0010560044990313,9.998751193898123,0.1999129343882228,1.0004776038280065,50.00744565711173,9.282486117003003
min,0,-10,Female,-6.21,70.62,-74.94,-4.96,-1.64,-0.57,-4.98,-161.28,5.47
max,3045569,89,Male,65.51,172.0,462.08,5.29,104.79,1.47,5.21,353.6,94.14


In [0]:
# Binarise progression score → Outcome (1 = above-median progression, 0 = below)
med_prog = df.approxQuantile("progression", [0.5], 0.01)[0]

# Replace biologically impossible zero values with column median
cols_with_zeros = ["glu", "bp", "bmi"]
medians = {c: df.approxQuantile(c, [0.5], 0.01)[0] for c in cols_with_zeros}

# Apply all columns at once — avoids deeply nested execution plan
df_clean = df.withColumns({
    "Outcome": when(col("progression") > med_prog, 1.0).otherwise(0.0),
    **{c: when(col(c) == 0, medians[c]).otherwise(col(c)) for c in cols_with_zeros}
})

# Verify zeros removed
display(df_clean.select([
    count(when(col(c) == 0, c)).alias(c)
    for c in cols_with_zeros]))

glu,bp,bmi
0,0,0


In [0]:
feature_cols = ["age", "bmi", "bp", "tc", "ldl", "hdl", "tch", "ltg", "glu"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features")

# 80% train, 20% test
train, test = df_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Train:{train.count()} Test:{test.count()}")

Train:2436615 Test:608955


In [0]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import mlflow.sklearn

mlflow.set_experiment('/diabetes_naivebayes_lab')

# Spark MLlib uses SQL higher-order functions blocked on Unity Catalog Serverless;
# train the equivalent Gaussian NaiveBayes via sklearn on pandas instead
train_pd = train.select(feature_cols + ["Outcome"]).toPandas()
test_pd  = test.select(feature_cols + ["Outcome"]).toPandas()

X_train, y_train = train_pd[feature_cols].values, train_pd["Outcome"].values
X_test,  y_test  = test_pd[feature_cols].values,  test_pd["Outcome"].values

nb = GaussianNB()

with mlflow.start_run(run_name='NaiveBayes_v1'):
    mlflow.log_param('modelType', 'gaussian')
    mlflow.log_param('smoothing', 1.0)

    nb.fit(X_train, y_train)
    y_pred = nb.predict(X_test)
    y_prob = nb.predict_proba(X_test)

    auc = roc_auc_score(y_test, y_prob[:, 1])
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)

    mlflow.log_metric('auc', auc)
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1', f1)
    mlflow.sklearn.log_model(nb, 'nb_diabetes_model')

    # Rebuild preds as a Spark DataFrame so Cell 7 continues to work
    test_pd["prediction"] = y_pred.astype(float)
    test_pd["probability"] = [row.tolist() for row in y_prob]
    preds = spark.createDataFrame(
        test_pd[["Outcome", "prediction", "probability"]])

    print(f'AUC: {auc:.4f}')
    print(f'Accuracy: {acc:.4f}')
    print(f'F1: {f1:.4f}')

2026/08/14 11:15:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-63d2a839-b046.cloud.databricks.com/ml/experiments/532190200800421/models/m-e04837c79c6d4db39608aedc81fb3c46?o=7474648888567396
2026/08/14 11:15:04 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


AUC: 0.9881
Accuracy: 0.9381
F1: 0.9384


In [0]:
cm = preds.groupBy("Outcome", "prediction").count()
display(cm.orderBy('Outcome', 'prediction'))

correct = preds.filter(col("Outcome") == col("prediction")).count()
total = preds.count()
print(f"Correct: {correct}/{total}")

display(preds.select("Outcome", "prediction", "probability"))

Outcome,prediction,count
0.0,0.0,284139
0.0,1.0,18900
1.0,0.0,18786
1.0,1.0,287130


Correct: 571269/608955


Outcome,prediction,probability
1.0,1.0,"List(0.1879890495841919, 0.8120109504158067)"
1.0,1.0,"List(0.25075048196547556, 0.7492495180345236)"
0.0,0.0,"List(0.8374501441274987, 0.16254985587249962)"
0.0,0.0,"List(0.9694507752725544, 0.030549224727446693)"
0.0,0.0,"List(0.5596165240941051, 0.44038347590589566)"
0.0,0.0,"List(0.9029462566376292, 0.09705374336237237)"
1.0,1.0,"List(0.15767577750005796, 0.8423242224999412)"
1.0,1.0,"List(0.08084124929205623, 0.9191587507079424)"
1.0,1.0,"List(0.03485707102130946, 0.9651429289786894)"
0.0,0.0,"List(0.9738326719071114, 0.026167328092887306)"


In [0]:
import pandas as pd

new = pd.DataFrame({
    'age':  [29, 50, 31],
    'bmi':  [28.1, 33.6, 26.6],
    'bp':   [68.0, 72.0, 66.0],
    'tc':   [140.0, 220.0, 130.0],
    'ldl':  [90.0, 130.0, 80.0],
    'hdl':  [55.0, 48.0, 60.0],
    'tch':  [0.7, 0.8, 0.6],
    'ltg':  [4.5, 5.1, 4.2],
    'glu':  [95.0, 148.0, 85.0]
})

new["prediction"] = nb.predict(new[feature_cols].values)
new["probability"] = [row.tolist() for row in nb.predict_proba(new[feature_cols].values)]
display(new[['glu', 'age', 'prediction', 'probability']])

glu,age,prediction,probability
95.0,29,1.0,"List(2.933987500806884E-6, 0.999997066012595)"
148.0,50,1.0,"List(6.39780795446851E-13, 1.0)"
85.0,31,1.0,"List(1.2899449624516285E-5, 0.9999871005502226)"
